# **TB-AI RAG Evaluation Notebook Using Ragaas**

I will be evaluating the TB-AI RAG system using **RAGAS** metrics.

**Steps to follow**
1. Load PDFs and chunk them
2. Generate synthetic QA pairs using Gemini
3. Run the RAG pipeline on the questions
4. Evaluate with RAGAS (Faithfulness, Context Recall, Context Precision, Response Relevancy)
5. Visualize results (To be Added)
6. Add LLM as a Judge(TO be Added)
7. Try both DeepEVal and Ragas at the same time(To be added Hopefully)



## **Mount Google Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Install Dependencies**

In [3]:
!pip install -q -U langchain-google-genai langchain-community sentence-transformers chromadb pdfplumber tqdm pandas plotly ragas datasets langchain langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/

## **Importing Stuff**

In [4]:
import os
import json
import random
import pdfplumber
import pandas as pd
from tqdm.auto import tqdm
from typing import List, Dict, Tuple

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from sentence_transformers import SentenceTransformer
import chromadb

In [5]:
# Set API KEy
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    os.environ["GOOGLE_API_KEY"] = input("Enter your Google API Key: ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

embedding_model = SentenceTransformer('All-MiniLM-L6-V2')
print("LLM and embedding model ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM and embedding model ready.


## **Load Knowledge Base**


In [6]:
def load_pdfs_from_folder(folder_path: str) -> List[Document]:
    documents = []
    pdf_files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]

    for pdf_file in tqdm(pdf_files, desc="Loading PDFs"):
        file_path = os.path.join(folder_path, pdf_file)
        try:
            with pdfplumber.open(file_path) as pdf:
                for i, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text and text.strip():
                        metadata = {
                            "SOURCE": pdf_file,
                            "PDF NAME": pdf_file,
                            "page_number": i + 1
                        }
                        documents.append(Document(page_content=text, metadata=metadata))
        except Exception as e:
            print(f"Error loading {pdf_file}: {e}")

    return documents

# Path to source_pdfs folder on Google Drive
SOURCE_PATH = "/content/drive/MyDrive/TB-AI/source_pdfs"
raw_docs = load_pdfs_from_folder(SOURCE_PATH)
print(f"\nLoaded {len(raw_docs)} pages from {len(set(doc.metadata['SOURCE'] for doc in raw_docs))} PDFs.")

Loading PDFs:   0%|          | 0/9 [00:00<?, ?it/s]


Loaded 1442 pages from 9 PDFs.


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

docs_processed = text_splitter.split_documents(raw_docs)
print(f"Created {len(docs_processed)} chunks.")

Created 2173 chunks.


## **Generate QA Pairs**
I will be using Gemini to create factoid question-answer pairs from random chunks.

In [8]:
QA_GENERATION_PROMPT = """
Your task is to write a factoid question and an answer given a context about Tuberculosis (TB).
Your factoid question should be answerable with a specific, concise piece of factual information from the context.
Do NOT mention "according to the passage" or "context".

Provide your answer as follows:

Output:::
Factoid question: (your factoid question)
Answer: (your answer to the factoid question)

Context: {context}
Output:::"""

def generate_qa_pairs(chunks: List[Document], n: int = 20) -> List[Dict]:
    #Generate QA pairs from random document chunks.
    outputs = []
    sampled = random.sample(chunks, min(n, len(chunks)))

    for chunk in tqdm(sampled, desc="Generating QA pairs"):
        prompt = QA_GENERATION_PROMPT.format(context=chunk.page_content)
        res = llm.invoke(prompt).content

        try:
            question = res.split("Factoid question:")[-1].split("Answer:")[0].strip()
            answer = res.split("Answer:")[-1].strip()
            outputs.append({
                "context": chunk.page_content,
                "question": question,
                "answer": answer,
                "source_doc": chunk.metadata['SOURCE']
            })
        except:
            continue

    return outputs

N_GENERATIONS = 10
qa_dataset = generate_qa_pairs(docs_processed, n=N_GENERATIONS)
print(f"Generated {len(qa_dataset)} QA pairs.")

Generating QA pairs:   0%|          | 0/10 [00:00<?, ?it/s]

Generated 10 QA pairs.


## **RAG System Setup**
Set up ChromaDB retrieval and the RAG answer function.

In [9]:
client = chromadb.PersistentClient(path="eval_vector_store")
collection = client.get_or_create_collection("TB_PDFs_Evaluation")

# Populate collection if empty
if collection.count() == 0:
    for i, doc in enumerate(tqdm(docs_processed, desc="Indexing chunks")):
        emb = embedding_model.encode([doc.page_content]).tolist()[0]
        collection.add(
            ids=[f"doc_{i}"],
            embeddings=[emb],
            documents=[doc.page_content],
            metadatas=[doc.metadata]
        )

print(f"Vector store has {collection.count()} chunks.")

Indexing chunks:   0%|          | 0/2173 [00:00<?, ?it/s]

Vector store has 2173 chunks.


In [10]:
def get_rag_context(query: str) -> Tuple[str, List[str]]:
    #Retrieve relevant chunks for a query.
    query_emb = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=5)
    context = "\n\n".join(results["documents"][0])
    return context, results["documents"][0]


QA_TEMPLATE = """You are a Tuberculosis RAG chatbot.
Answer the user query ONLY using the provided context.
Context: {context}
Question: {question}
Answer:"""


def answer_with_rag(question: str) -> Dict:
    """Get RAG answer for a question."""
    context_text, context_chunks = get_rag_context(question)
    prompt = QA_TEMPLATE.format(context=context_text, question=question)
    response = llm.invoke(prompt).content
    return {"response": response, "context": context_chunks}

# Quick test
test = answer_with_rag("What is tuberculosis?")
print("RAG test:", test["response"][:200])

RAG test: Tuberculosis (TB) is one of the oldest infectious diseases in the world and it can affect almost any tissue or organ of the body.


## **6. Run RAG on QA Dataset**

In [11]:
test_results = []
for item in tqdm(qa_dataset, desc="Testing RAG"):
    rag_output = answer_with_rag(item['question'])
    test_results.append({
        "question": item['question'],
        "ground_truth": item['answer'],
        "answer": rag_output['response'],
        "contexts": rag_output['context']
    })

results_df = pd.DataFrame(test_results)
print(f"RAG testing complete. {len(results_df)} results.")
results_df.head()

Testing RAG:   0%|          | 0/10 [00:00<?, ?it/s]

RAG testing complete. 10 results.


,question,ground_truth,answer,contexts
0,How are catastrophic costs defined for TB pati...,Direct medical and non-medical costs plus inco...,"In the context of TB patient cost surveys, cat...",[(https://ghcosting.org/pages/standards/refere...
1,Which high TB burden countries have implemente...,"China, Kenya, Pakistan, and Viet Nam.","The provided context states, ""The number of in...",[Annex 2\nCountry profiles\nFOR 30 HIGH TB BUR...
2,What are the two principal strategies for cons...,The Bayer and the Gould-Jacobs routes.,The two principal strategies for the construct...,[Promising Drug Candidates in Advanced Tubercu...
3,How often should long-term inmates and correct...,At least annually.,Long-term inmates and correctional facility st...,[Table 6.5\nTB Infection Control for Non-tradi...
4,Who reviewed Chapter 3 of the report?,Jessica Ho,Chapter 3 (TB disease burden) was prepared by ...,"[Ahmedov, Amna Al-Gallas-Streeter, Kenneth Cas..."


## **RAGAS Evaluation**
Evaluate using RAGAS metrics:
- **Faithfulness**: Is the answer grounded in the retrieved context?
- **Response Relevancy**: Is the answer relevant to the question?
- **Context Precision**: Are the retrieved contexts relevant?
- **Context Recall**: Does the context contain the needed info?



In [13]:
!pip install -q -U langchain-huggingface

In [16]:
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

# Wrap the Gemini LLM for RAGAS
ragas_llm = LangchainLLMWrapper(llm)

# Use All-MiniLM-L6-V2 as the RAGAS embedding model
hf_embeddings = HuggingFaceEmbeddings(model_name="All-MiniLM-L6-V2")
ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

print("RAGAS wrappers ready.")

/tmp/ipykernel_13950/2313031422.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall
/tmp/ipykernel_13950/2313031422.py:2: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall
/tmp/ipykernel_13950/2313031422.py:2: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collection

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/All-MiniLM-L6-V2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAGAS wrappers ready.


/tmp/ipykernel_13950/2313031422.py:12: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)


In [17]:
# Build RAGAS EvaluationDataset from our test results
samples = []
for _, row in results_df.iterrows():
    sample = SingleTurnSample(
        user_input=row["question"],
        response=row["answer"],
        retrieved_contexts=row["contexts"],
        reference=row["ground_truth"]
    )
    samples.append(sample)

eval_ds = EvaluationDataset(samples=samples)
print(f"Created EvaluationDataset with {len(eval_ds)} samples.")

Created EvaluationDataset with 10 samples.


In [18]:
# Define metrics
metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    LLMContextRecall(llm=ragas_llm),
]

# Run evaluation
ragas_result = evaluate(
    dataset=eval_ds,
    metrics=metrics,
)

print("\n=== RAGAS Results ===")
print(ragas_result)

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[1]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[2]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[4]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[5]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[6]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[7]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[10]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[11]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[12]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[13]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[14]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[15]: TimeoutError()
ERROR:ragas.executor:Exception rai


=== RAGAS Results ===
{'faithfulness': nan, 'answer_relevancy': 0.9944, 'llm_context_precision_without_reference': nan, 'context_recall': nan}
